# Facial Emotion Recognition — Combined Training (FER+ Balanced + RAF-DB Balanced)

**Goal:** train ONE robust EfficientNet-B3 + CBAM model on the two independent balanced
datasets combined (FER+ Balanced and RAF-DB Balanced),
usable directly with the Streamlit webcam app (same architecture, same label order).

**Design choices made to avoid overfitting / data leakage (per your requirements):**

1. Each dataset's *original* train/test split is preserved — we concatenate splits of the
   same type across datasets instead of merging everything and re-splitting randomly.
   Re-splitting after merging is a common cause of leakage in FER datasets that contain
   duplicated/augmented images.
2. An exact-duplicate hash check removes any test image whose file content also appears in
   train (leakage caused by "balanced" forks that oversample minority classes before splitting).
   NOTE: we intentionally do NOT combine the raw `msambare/fer2013` dataset here — the
   "Balanced FER-Dataset (75x75)" is itself built FROM FER2013 (same source images, just
   rebalanced + resized), so adding both would silently duplicate the same faces across
   "two different" datasets, undetectable by exact-hash dedup since resizing changes the
   file bytes. Using only the balanced version avoids this hidden leakage entirely.
3. The combined test pool is further split into a **validation** set (used only for early
   stopping / LR scheduling) and a small **held-out test** set that is *never* used for
   model selection — the number we report at the end is an honest, unbiased estimate.
4. Two-phase fine-tuning (short warm-up with the backbone frozen, then full fine-tuning)
   avoids destroying the pretrained ImageNet features early on.
5. Standard regularization kept: weighted loss (combined pool isn't perfectly balanced),
   label smoothing, dropout, weight decay, strong augmentation, early stopping.

Just run all cells top to bottom on Kaggle (GPU accelerator ON, Internet not required).

In [ ]:
import os
import cv2
import time
import random
import hashlib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import confusion_matrix, classification_report

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU, enable it from Settings -> Accelerator")

## Step 0 — Dataset paths & label mapping
Canonical emotion order matches the already-built Streamlit app exactly.

In [ ]:
# ------------------------------------------------------------------
# Paths (exactly as attached on Kaggle)
# ------------------------------------------------------------------
# NOTE: msambare/fer2013 is intentionally NOT included — the balanced FER
# dataset below is already built from FER2013 itself (same source photos,
# just rebalanced + resized). Adding both would duplicate the same faces
# under two different dataset names.
DATASET_ROOTS = {
    'ferplus_balanced': '/kaggle/input/datasets/dollyprajapati182/balanced-fer-dataset-7575-grayscale',
    'rafdb_balanced':   '/kaggle/input/datasets/dollyprajapati182/balanced-raf-db-dataset-7575-grayscale',
}

CANONICAL_EMOTIONS = ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']
EMOTION_TO_IDX = {e: i for i, e in enumerate(CANONICAL_EMOTIONS)}

SYNONYMS = {
    'angry': 'Angry', 'anger': 'Angry',
    'disgust': 'Disgust', 'disgusted': 'Disgust',
    'fear': 'Fear', 'fearful': 'Fear', 'afraid': 'Fear',
    'happy': 'Happy', 'happiness': 'Happy', 'joy': 'Happy',
    'neutral': 'Neutral', 'natural': 'Neutral',
    'sad': 'Sad', 'sadness': 'Sad',
    'surprise': 'Surprise', 'surprised': 'Surprise', 'shock': 'Surprise',
}




def map_folder_to_canonical(folder_name):
    key = folder_name.strip().lower()
    if key in SYNONYMS:
        return SYNONYMS[key]
    for k, v in SYNONYMS.items():
        if key.startswith(k) or k.startswith(key):
            return v
    return None  # unknown class (e.g. "contempt") -> skipped




def resolve_split_dirs(root):
    """Finds the train/test subfolders inside a dataset root, tolerant to naming variants."""
    entries = [d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]
    lower_map = {d.lower(): d for d in entries}

    train_names = ['train', 'training']
    test_names = ['test', 'testing', 'val', 'validation']

    train_dir = next((os.path.join(root, lower_map[n]) for n in train_names if n in lower_map), None)
    test_dir = next((os.path.join(root, lower_map[n]) for n in test_names if n in lower_map), None)

    if train_dir is None or test_dir is None:
        # maybe nested one level deeper
        for d in entries:
            sub = os.path.join(root, d)
            if os.path.isdir(sub):
                sub_lower = [x.lower() for x in os.listdir(sub)]
                if any(n in sub_lower for n in train_names):
                    return resolve_split_dirs(sub)
        raise FileNotFoundError(
            f"Could not find train/test folders inside '{root}'. Found: {entries}"
        )
    return train_dir, test_dir




def scan_dataset(root_dir):
    """Returns list of (image_path, canonical_label) for one split folder."""
    samples = []
    for folder in sorted(os.listdir(root_dir)):
        folder_path = os.path.join(root_dir, folder)
        if not os.path.isdir(folder_path):
            continue
        canonical = map_folder_to_canonical(folder)
        if canonical is None:
            print(f"  Skipping unmapped class folder: '{folder}'")
            continue
        for fname in os.listdir(folder_path):
            if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                samples.append((os.path.join(folder_path, fname), canonical))
    return samples

## Step 1 — Scan both datasets (original splits kept intact)

In [ ]:
# ------------------------------------------------------------------
# Step 1: scan both datasets, keeping ORIGINAL train/test splits intact.
# We do NOT merge-then-reshuffle: that would risk leaking near-duplicate /
# augmented images across train and test. Instead we concatenate each
# dataset's own train split into one big train pool, and each dataset's
# own test split into one big test pool.
# ------------------------------------------------------------------
train_samples = []
test_samples = []

print("=" * 70)
print("SCANNING DATASETS")
print("=" * 70)
for name, root in DATASET_ROOTS.items():
    if not os.path.exists(root):
        raise FileNotFoundError(f"Dataset '{name}' not found at {root}. Check it is attached to this notebook.")
    train_dir, test_dir = resolve_split_dirs(root)
    ds_train = scan_dataset(train_dir)
    ds_test = scan_dataset(test_dir)
    train_samples.extend(ds_train)
    test_samples.extend(ds_test)
    print(f"{name:18s}: train={len(ds_train):6,}  test={len(ds_test):6,}")

print(f"\nCombined BEFORE dedup -> train={len(train_samples):,}  test={len(test_samples):,}")

## Step 2 — Deduplication & leakage removal

In [ ]:
# ------------------------------------------------------------------
# Step 2: remove exact duplicate images (same file content) to avoid
# double-counting and, more importantly, remove any TEST image whose
# exact content also appears in TRAIN (a common source of inflated /
# unrealistic accuracy on Kaggle "balanced" forks that oversample by
# duplicating images before splitting).
# ------------------------------------------------------------------
def hash_file(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()


def dedup_samples(samples, seen_hashes=None):
    """Drops samples whose file hash was already seen. Returns (kept, hashes)."""
    if seen_hashes is None:
        seen_hashes = set()
    kept = []
    for path, label in tqdm(samples, desc="Hashing"):
        h = hash_file(path)
        if h in seen_hashes:
            continue
        seen_hashes.add(h)
        kept.append((path, label))
    return kept, seen_hashes


print("\n" + "=" * 70)
print("DEDUPLICATION (removing internal duplicates + train/test leakage)")
print("=" * 70)

train_samples, train_hashes = dedup_samples(train_samples)
before_test = len(test_samples)
test_samples, _ = dedup_samples(test_samples, seen_hashes=set(train_hashes))
removed_leak = before_test - len(test_samples)

print(f"\nTrain (deduplicated): {len(train_samples):,}")
print(f"Test  (deduplicated, leak-free): {len(test_samples):,}")
print(f"Test images removed because they duplicated a train image: {removed_leak:,}")

## Step 3 — Validation / held-out test split (stratified)

In [ ]:
# ------------------------------------------------------------------
# Step 3: carve a small held-out TEST set out of the combined test pool,
# keeping the rest as VALIDATION (used only for early stopping / LR
# scheduling). This way the number we report at the end was never used
# to pick the checkpoint, which is a more honest measure of generalization.
# ------------------------------------------------------------------
def stratified_split(samples, holdout_ratio=0.25, seed=42):
    rng = random.Random(seed)
    by_class = {}
    for path, label in samples:
        by_class.setdefault(label, []).append((path, label))
    val, holdout = [], []
    for label, items in by_class.items():
        rng.shuffle(items)
        n_holdout = max(1, int(len(items) * holdout_ratio))
        holdout.extend(items[:n_holdout])
        val.extend(items[n_holdout:])
    rng.shuffle(val)
    rng.shuffle(holdout)
    return val, holdout


val_samples, holdout_test_samples = stratified_split(test_samples, holdout_ratio=0.25)
print(f"\nValidation samples (for early stopping): {len(val_samples):,}")
print(f"Held-out test samples (final report only): {len(holdout_test_samples):,}")

# Combined class distribution (sanity check before training)
train_dist = pd.Series([l for _, l in train_samples]).value_counts().reindex(CANONICAL_EMOTIONS)
print("\nCombined TRAIN class distribution:")
print(train_dist)

## Preprocessing, Dataset class, Transforms & DataLoaders

In [ ]:
# ------------------------------------------------------------------
# Preprocessing (identical to the original training pipeline / app)
# ------------------------------------------------------------------
def preprocess_image(image):
    image = cv2.bilateralFilter(image, 5, 50, 50)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    image = clahe.apply(image)
    mean_intensity = np.mean(image)
    gamma = 1.5 if mean_intensity < 100 else (0.8 if mean_intensity > 155 else 1.2)
    table = np.array([((i / 255.0) ** (1.0 / gamma)) * 255 for i in range(256)]).astype("uint8")
    return cv2.LUT(image, table)




class CombinedFERDataset(Dataset):
    """Dataset built from a list of (image_path, canonical_label) tuples."""

    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        image = preprocess_image(image)
        image = cv2.resize(image, (224, 224))
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        image = Image.fromarray(image)
        if self.transform:
            image = self.transform(image)
        return image, EMOTION_TO_IDX[label]




train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.RandomAffine(degrees=0, translate=(0.2, 0.2), scale=(0.85, 1.15)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.4, scale=(0.02, 0.15)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

BATCH_SIZE = 32

train_dataset = CombinedFERDataset(train_samples, transform=train_transforms)
val_dataset = CombinedFERDataset(val_samples, transform=eval_transforms)
holdout_test_dataset = CombinedFERDataset(holdout_test_samples, transform=eval_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
holdout_test_loader = DataLoader(holdout_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Class weights (combined pool is not perfectly balanced even though the
# individual "balanced" datasets were, because FER2013-original is imbalanced)
counts = [train_dist[e] for e in CANONICAL_EMOTIONS]
total = sum(counts)
class_weights = torch.FloatTensor([total / c for c in counts])
class_weights = (class_weights / class_weights.sum() * len(class_weights)).to(device)
print("\nClass weights:", dict(zip(CANONICAL_EMOTIONS, class_weights.cpu().numpy().round(2))))

## Model — EfficientNet-B3 + CBAM

In [ ]:
# ------------------------------------------------------------------
# Model: EfficientNet-B3 + CBAM (identical to model.py used by the app)
# ------------------------------------------------------------------
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()
        avg = self.fc(self.avg_pool(x).view(b, c))
        max_ = self.fc(self.max_pool(x).view(b, c))
        return x * self.sigmoid(avg + max_).view(b, c, 1, 1).expand_as(x)


class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, 7, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        max_, _ = torch.max(x, dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg, max_], dim=1)))


class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention()

    def forward(self, x):
        return self.sa(self.ca(x))


import socket
socket.setdefaulttimeout(15)  # fail fast instead of hanging if there's no internet


def create_model(num_classes=7):
    try:
        model = models.efficientnet_b3(weights='IMAGENET1K_V1')
        print("Loaded ImageNet-pretrained EfficientNet-B3 weights.")
    except Exception as e:
        print(f"Could not download pretrained weights ({e}).")
        print("Falling back to RANDOM initialization instead of hanging.")
        print("For best accuracy, enable Internet in Notebook Settings and rerun.")
        model = models.efficientnet_b3(weights=None)
    model.features = nn.Sequential(model.features, CBAM(1536))
    num_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Linear(num_features, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(512, num_classes),
    )
    return model


model = create_model(num_classes=7).to(device)
print(f"\nTotal params: {sum(p.numel() for p in model.parameters()):,}")

## Train / evaluate functions

In [ ]:
# ------------------------------------------------------------------
# Train / eval loops
# Mixed precision (torch.autocast + GradScaler) speeds up training on GPU
# by ~30-50% and reduces memory, with no accuracy downside. Pass a
# GradScaler to enable it; leave scaler=None to train in plain fp32.
# ------------------------------------------------------------------
def train_one_epoch(model, loader, criterion, optimizer, device, scaler=None):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    use_amp = scaler is not None
    pbar = tqdm(loader, desc="Training")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)
        if use_amp:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix(loss=f"{running_loss/total:.4f}", acc=f"{100*correct/total:.1f}%")
    return running_loss / total, 100 * correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating"):
            images, labels = images.to(device), labels.to(device)
            with torch.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                outputs = model(images)
                loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return running_loss / total, 100 * correct / total, all_preds, all_labels

## Training — Phase 1 (warm-up) then Phase 2 (full fine-tune) with early stopping

In [ ]:
SAVE_PATH = '/kaggle/working/best_model_combined.pth'
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

# Mixed precision scaler (only active on GPU; harmless no-op on CPU)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

# ---- Phase 1: warm-up the new head only (backbone frozen) ----
# This avoids destroying the pretrained ImageNet features with large
# random-init gradients from the new classifier head at the very start,
# which is a common cause of early overfitting when fine-tuning.
for param in model.features[0].parameters():
    param.requires_grad = False

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)
print("\n" + "=" * 70)
print("PHASE 1: warm-up (backbone frozen, head + CBAM only) — 3 epochs")
print("=" * 70)
for epoch in range(3):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler=scaler)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
    print(f"Warm-up epoch {epoch+1}/3 | Train {train_acc:.2f}% | Val {val_acc:.2f}%")

# ---- Phase 2: fine-tune the whole network ----
for param in model.features[0].parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

best_acc = 0.0
patience_counter = 0
MAX_PATIENCE = 10
NUM_EPOCHS = 40
history = {'train_acc': [], 'val_acc': []}

print("\n" + "=" * 70)
print("PHASE 2: full fine-tuning")
print("=" * 70)
for epoch in range(NUM_EPOCHS):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler=scaler)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)

    old_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_acc)
    new_lr = optimizer.param_groups[0]['lr']

    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    gap = train_acc - val_acc
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS} | Train {train_acc:.2f}% | Val {val_acc:.2f}% "
          f"| Gap {gap:.1f}pt | {time.time()-t0:.0f}s")
    if gap > 15:
        print("  NOTE: large train/val gap — possible overfitting, watch next epochs")
    if old_lr != new_lr:
        print(f"  LR: {old_lr:.2e} -> {new_lr:.2e}")

    if val_acc > best_acc:
        best_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  NEW BEST: {best_acc:.2f}% -> saved to {SAVE_PATH}")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{MAX_PATIENCE})")
        if patience_counter >= MAX_PATIENCE:
            print("  Early stopping.")
            break

print(f"\nBest validation accuracy: {best_acc:.2f}%")

## Temperature Scaling — calibrating the model's confidence

Neural networks are typically **overconfident**: they might say "98% Happy" when the
true reliability of that confidence is closer to 80%. This matters directly for our app's
`CONFIDENCE_THRESHOLD = 0.40` rule ("show Uncertain below this") — an uncalibrated model
makes that threshold less meaningful.

Temperature Scaling fixes this with a single learned scalar `T`: we divide the logits by
`T` before the softmax (`softmax(logits / T)`). `T` is fit on the **validation set only**
(never on held-out test) by minimizing NLL loss with the model itself kept frozen — it's a
cheap, standard post-hoc calibration step (Guo et al., 2017).

In [ ]:
import json

def collect_logits(model, loader, device):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Collecting logits"):
            images = images.to(device)
            with torch.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits = model(images)
            all_logits.append(logits.float().cpu())
            all_labels.append(labels)
    return torch.cat(all_logits), torch.cat(all_labels)


def fit_temperature(logits, labels, device, max_iter=50):
    logits = logits.to(device)
    labels = labels.to(device)
    log_temperature = torch.zeros(1, requires_grad=True, device=device)
    nll_criterion = nn.CrossEntropyLoss()
    optimizer = optim.LBFGS([log_temperature], lr=0.05, max_iter=max_iter)

    def closure():
        optimizer.zero_grad()
        temperature = torch.exp(log_temperature)
        loss = nll_criterion(logits / temperature, labels)
        loss.backward()
        return loss

    optimizer.step(closure)
    return torch.exp(log_temperature).item()


print("Loading best checkpoint for calibration...")
model.load_state_dict(torch.load(SAVE_PATH))

val_logits, val_labels = collect_logits(model, val_loader, device)
TEMPERATURE = fit_temperature(val_logits, val_labels, device)
print(f"\nLearned temperature: T = {TEMPERATURE:.3f}")
print("(T > 1 means the model was overconfident and probabilities are now softened;"
      " T ~= 1 means it was already well-calibrated)")

with open('/kaggle/working/temperature.json', 'w') as f:
    json.dump({"temperature": TEMPERATURE}, f)
print("Saved to /kaggle/working/temperature.json — copy this next to app.py too.")

## Final report on the held-out test set (unbiased) + overfitting check

Two numbers are reported: the plain evaluation (same as before), and the **TTA + calibrated** evaluation — each image is predicted normally AND on its horizontal mirror, the two probability vectors are averaged, and the temperature learned above is applied. TTA alone typically adds ~1-2% accuracy for free (no retraining needed); it's exactly what the app's *image upload* mode does internally.

In [ ]:
# ------------------------------------------------------------------
# Final, unbiased report on the held-out test set
# ------------------------------------------------------------------
def evaluate_with_tta(model, loader, device, temperature=1.0):
    """Averages predictions on the original image and its horizontal flip,
    then applies temperature scaling. Same trick the app uses on uploaded images."""
    model.eval()
    correct, total = 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating (TTA)"):
            images, labels = images.to(device), labels.to(device)
            flipped = torch.flip(images, dims=[3])
            with torch.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits_orig = model(images)
                logits_flip = model(flipped)
            probs = (F.softmax(logits_orig.float() / temperature, dim=1) +
                     F.softmax(logits_flip.float() / temperature, dim=1)) / 2
            preds = probs.argmax(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return 100 * correct / total, all_preds, all_labels


model.load_state_dict(torch.load(SAVE_PATH))
test_loss, test_acc, _, _ = evaluate(model, holdout_test_loader, criterion, device)
print(f"\nHELD-OUT TEST ACCURACY (plain, no TTA): {test_acc:.2f}%")

tta_acc, test_preds, test_labels = evaluate_with_tta(model, holdout_test_loader, device, temperature=TEMPERATURE)
print(f"HELD-OUT TEST ACCURACY (TTA + calibrated, T={TEMPERATURE:.2f}): {tta_acc:.2f}%")
test_acc = tta_acc  # use the improved number for the plots/report below

cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CANONICAL_EMOTIONS, yticklabels=CANONICAL_EMOTIONS, linewidths=0.5)
plt.title(f'Confusion Matrix — Combined Dataset (Held-out Test)\nAccuracy: {test_acc:.2f}%')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix_combined.png', dpi=300)
plt.show()

print(classification_report(test_labels, test_preds, target_names=CANONICAL_EMOTIONS, digits=4))

plt.figure(figsize=(10, 6))
plt.plot(history['train_acc'], label='Train Accuracy')
plt.plot(history['val_acc'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Train vs Validation Accuracy (overfitting check)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/train_val_curve.png', dpi=300)
plt.show()

print("\nSaved files in /kaggle/working/:")
for f in ['best_model_combined.pth', 'confusion_matrix_combined.png', 'train_val_curve.png']:
    p = f'/kaggle/working/{f}'
    if os.path.exists(p):
        print(f"  OK {f} ({os.path.getsize(p)/1e6:.2f} MB)")
    else:
        print(f"  MISSING {f}")